# XLK Initial Feature Engineering

This notebook constructs a compact, interpretable and leakage-safe initial predictor set for the existing next-five-trading-day high-volatility target. It uses only project data produced by the feasibility notebook. No model is trained, and the existing target, threshold, chronological split and five-observation purge are preserved.

In [ ]:
from pathlib import Path
import os

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'src').is_dir() and (candidate / 'notebooks').is_dir():
            return candidate
    raise RuntimeError('Run this notebook from inside the cloned repository.')

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)

os.environ.setdefault('MPLCONFIGDIR', str(PROJECT_ROOT / '.venv' / '.matplotlib'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RAW_PATH = PROJECT_ROOT / 'data' / 'raw' / 'xlk_daily_2000_2026.csv'
FEASIBILITY_PATH = PROJECT_ROOT / 'data' / 'processed' / 'xlk_feasibility_dataset.csv'
SUMMARY_PATH = PROJECT_ROOT / 'outputs' / 'tables' / 'xlk_feasibility_summary.csv'
FEATURE_PATH = PROJECT_ROOT / 'data' / 'processed' / 'xlk_feature_dataset.csv'
DICTIONARY_PATH = PROJECT_ROOT / 'outputs' / 'tables' / 'xlk_feature_dictionary.csv'
HEATMAP_PATH = PROJECT_ROOT / 'outputs' / 'figures' / 'xlk_training_feature_correlation.png'

for path in [RAW_PATH, FEASIBILITY_PATH, SUMMARY_PATH]:
    if not path.is_file():
        raise FileNotFoundError(f'Required project input is missing: {path}')

sns.set_theme(style='whitegrid')
print(f'Confirmed project root: {PROJECT_ROOT}')
print('All three required project inputs are available; no external data will be requested.')

## 1. Input reconstruction and validation

The raw yfinance CSV contains two column-header levels (`Price` and `Ticker`) plus a named date-index row. It is therefore read with two explicit header rows while skipping the separate index-name row. The XLK ticker level is validated before it is removed for analysis. The reconstructed observations are then checked against the feasibility summary and processed feasibility dataset.

In [ ]:
raw_multi = pd.read_csv(
    RAW_PATH, header=[0, 1], skiprows=[2], index_col=0, parse_dates=[0]
)
if not isinstance(raw_multi.columns, pd.MultiIndex) or raw_multi.columns.nlevels != 2:
    raise RuntimeError('The saved raw file does not have the expected two-level yfinance column structure.')
if raw_multi.columns.names != ['Price', 'Ticker']:
    raise RuntimeError(f'Unexpected MultiIndex level names: {raw_multi.columns.names}')
ticker_values = raw_multi.columns.get_level_values('Ticker').unique().tolist()
if set(ticker_values) != {'XLK'}:
    raise RuntimeError(f'Unexpected ticker values: {ticker_values}')

ohlcv = raw_multi.xs('XLK', axis=1, level='Ticker', drop_level=True).copy()
ohlcv.index = pd.DatetimeIndex(ohlcv.index, name='Date')
expected_columns = ['Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume']
if ohlcv.columns.tolist() != expected_columns:
    raise RuntimeError(f'Unexpected reconstructed columns: {ohlcv.columns.tolist()}')

feasibility = pd.read_csv(FEASIBILITY_PATH, parse_dates=['Date'])
summary_table = pd.read_csv(SUMMARY_PATH, dtype=str)
if summary_table['metric'].duplicated().any():
    raise RuntimeError('The feasibility summary contains duplicated metric names.')
summary = summary_table.set_index('metric')['value'].to_dict()

assert ohlcv.index.is_monotonic_increasing and ohlcv.index.is_unique, \
    'Reconstructed OHLCV dates must be increasing and unique.'
assert len(ohlcv) == int(summary['raw_sample_size']), \
    'The reconstructed raw sample size does not match the feasibility summary.'
assert ohlcv.index.min().date().isoformat() == summary['first_observation_date'], \
    'The reconstructed first date does not match the feasibility summary.'
assert ohlcv.index.max().date().isoformat() == summary['last_observation_date'], \
    'The reconstructed last date does not match the feasibility summary.'
assert not ohlcv.isna().any().any(), 'The reconstructed OHLCV data contain missing values.'
assert feasibility['Date'].is_monotonic_increasing and feasibility['Date'].is_unique, \
    'Feasibility dates must be increasing and unique.'
assert len(feasibility) == int(summary['final_modelling_sample_size']), \
    'The feasibility dataset size does not match its summary.'

purged_dates = pd.to_datetime([date.strip() for date in summary['purged_observation_dates'].split(';')])
expected_test_start = pd.Timestamp('2021-03-10')
actual_test_start = feasibility.loc[feasibility['sample_period'].eq('test'), 'Date'].min()
assert len(purged_dates) == 5 and not feasibility['Date'].isin(purged_dates).any(), \
    'The existing five purged dates must remain absent.'
assert actual_test_start == expected_test_start, 'The original test start date must remain 10 March 2021.'
assert feasibility['sample_period'].value_counts().to_dict() == {
    'train': int(summary['final_training_sample_size_after_purging']),
    'test': int(summary['unchanged_test_sample_size']),
}, 'Train and test sample sizes do not match the feasibility summary.'

print('The yfinance MultiIndex structure was reconstructed explicitly and validated.')
print(f'Reconstructed OHLCV range: {ohlcv.index.min().date()} to {ohlcv.index.max().date()}')
print(f'Reconstructed OHLCV observations: {len(ohlcv):,}')
print(f'Existing feasibility observations: {len(feasibility):,}')
existing_train_size = int(feasibility['sample_period'].eq('train').sum())
existing_test_size = int(feasibility['sample_period'].eq('test').sum())
print(f'Existing train/test sizes: {existing_train_size:,} / {existing_test_size:,}')
print('Existing purged dates:', ', '.join(date.date().isoformat() for date in purged_dates))

## 2. Leakage-safe feature construction

Every rolling window ends at date *t*. No negative shift is used, and neither `future_rv_5d` nor `high_volatility` enters the feature-construction function.

The 20-day downside volatility is defined as

$$\sqrt{252}\sqrt{\frac{1}{20}\sum_{i=0}^{19}\min(r_{t-i},0)^2}.$$

Thus, only negative logarithmic returns contribute a non-zero magnitude; non-negative returns contribute zero.

The 14-day RSI uses adjusted-close changes: average gain is the arithmetic mean of positive changes over the latest 14 trading days, average loss is the arithmetic mean of the absolute values of negative changes, relative strength is average gain divided by average loss, and $RSI=100-100/(1+RS)$. If average loss is zero, RSI is set to 100 when average gain is positive and to 50 when both averages are zero.

In [ ]:
PREDICTORS = [
    'log_return_1d', 'return_5d', 'return_20d',
    'volatility_5d', 'volatility_20d', 'downside_volatility_20d',
    'price_to_ma_10', 'price_to_ma_50', 'rsi_14',
    'volume_ratio_20d', 'intraday_range',
]

def construct_predictors(frame):
    required = {'Adj Close', 'Close', 'High', 'Low', 'Volume'}
    if not required.issubset(frame.columns):
        raise ValueError(f'Missing required OHLCV fields: {sorted(required - set(frame.columns))}')
    if not (frame['Adj Close'] > 0).all() or not (frame['Close'] > 0).all():
        raise ValueError('Adjusted Close and Close must be positive before division.')

    result = pd.DataFrame(index=frame.index)
    log_return = np.log(frame['Adj Close'] / frame['Adj Close'].shift(1))
    result['log_return_1d'] = log_return
    result['return_5d'] = log_return.rolling(5, min_periods=5).sum()
    result['return_20d'] = log_return.rolling(20, min_periods=20).sum()
    result['volatility_5d'] = log_return.rolling(5, min_periods=5).std(ddof=1) * np.sqrt(252)
    result['volatility_20d'] = log_return.rolling(20, min_periods=20).std(ddof=1) * np.sqrt(252)
    downside_squared = log_return.clip(upper=0).pow(2)
    result['downside_volatility_20d'] = np.sqrt(downside_squared.rolling(20, min_periods=20).mean()) * np.sqrt(252)

    ma_10 = frame['Adj Close'].rolling(10, min_periods=10).mean()
    ma_50 = frame['Adj Close'].rolling(50, min_periods=50).mean()
    volume_mean_20 = frame['Volume'].rolling(20, min_periods=20).mean()
    assert (ma_10.dropna() > 0).all() and (ma_50.dropna() > 0).all(), \
        'Moving-average denominators must be positive.'
    assert (volume_mean_20.dropna() > 0).all(), 'Trailing volume denominators must be positive.'
    result['price_to_ma_10'] = frame['Adj Close'] / ma_10 - 1
    result['price_to_ma_50'] = frame['Adj Close'] / ma_50 - 1

    price_change = frame['Adj Close'].diff()
    gain = price_change.clip(lower=0)
    loss = -price_change.clip(upper=0)
    average_gain = gain.rolling(14, min_periods=14).mean()
    average_loss = loss.rolling(14, min_periods=14).mean()
    rsi = pd.Series(np.nan, index=frame.index, dtype=float)
    regular_loss = average_loss > 0
    relative_strength = average_gain.loc[regular_loss] / average_loss.loc[regular_loss]
    rsi.loc[regular_loss] = 100 - 100 / (1 + relative_strength)
    rsi.loc[average_loss.eq(0) & average_gain.gt(0)] = 100.0
    rsi.loc[average_loss.eq(0) & average_gain.eq(0)] = 50.0
    result['rsi_14'] = rsi

    result['volume_ratio_20d'] = frame['Volume'] / volume_mean_20
    result['intraday_range'] = (frame['High'] - frame['Low']) / frame['Close']
    return result[PREDICTORS]

features = construct_predictors(ohlcv)

for checkpoint in [100, len(ohlcv) // 2, len(ohlcv) - 1]:
    prefix_features = construct_predictors(ohlcv.iloc[:checkpoint + 1]).iloc[-1]
    pd.testing.assert_series_equal(
        prefix_features, features.iloc[checkpoint], check_names=False, rtol=1e-12, atol=1e-12
    )

assert features.index.equals(ohlcv.index), 'Feature dates must match the reconstructed raw dates.'
assert features.index.is_monotonic_increasing and features.index.is_unique, \
    'Feature dates must be increasing and unique.'
assert not np.isinf(features.to_numpy(dtype=float)).any(), 'Predictors must not contain infinite values.'
print('All predictors were constructed with windows ending at the prediction date.')
print('Causality checks using truncated histories passed at three points in the sample.')
print('Missing predictor values before merging, caused by initial lookbacks:')
print(features.isna().sum().to_string())

## 3. Feature dictionary

The dictionary records each initial predictor precisely. Annualisation uses 252 trading days where applicable.

In [ ]:
dictionary_rows = [
    ('log_return_1d', 'Return', 'ln(Adjusted Close_t / Adjusted Close_(t-1))', '2 prices / 1 return', 'No', 'Adjusted closes at t and t-1', 'One-day continuously compounded return'),
    ('return_5d', 'Return', 'Sum of log_return_1d from t-4 through t', '5 trading days', 'No', 'Returns ending at t', 'Short-horizon cumulative return'),
    ('return_20d', 'Return', 'Sum of log_return_1d from t-19 through t', '20 trading days', 'No', 'Returns ending at t', 'Medium-horizon cumulative return'),
    ('volatility_5d', 'Volatility and risk', 'Sample standard deviation of the latest 5 daily logarithmic returns × sqrt(252)', '5 trading days', 'Yes', 'Returns ending at t', 'Recent annualised return variability'),
    ('volatility_20d', 'Volatility and risk', 'Sample standard deviation of the latest 20 daily logarithmic returns × sqrt(252)', '20 trading days', 'Yes', 'Returns ending at t', 'Medium-horizon annualised return variability'),
    ('downside_volatility_20d', 'Volatility and risk', 'sqrt(mean(min(log_return, 0)^2) over 20 days) × sqrt(252)', '20 trading days', 'Yes', 'Returns ending at t', 'Annualised magnitude of recent negative returns'),
    ('price_to_ma_10', 'Momentum', 'Adjusted Close_t / mean(Adjusted Close from t-9 through t) − 1', '10 trading days', 'No', 'Adjusted closes ending at t', 'Distance above or below the short moving average'),
    ('price_to_ma_50', 'Momentum', 'Adjusted Close_t / mean(Adjusted Close from t-49 through t) − 1', '50 trading days', 'No', 'Adjusted closes ending at t', 'Distance above or below the medium moving average'),
    ('rsi_14', 'Momentum', '100 − 100 / (1 + average gain / average loss), using adjusted-close changes', '14 trading days', 'No', 'Adjusted-close changes ending at t', 'Momentum balance from 0 to 100'),
    ('volume_ratio_20d', 'Volume and range', 'Volume_t / mean(Volume from t-19 through t)', '20 trading days', 'No', 'Volumes ending at t', 'Current activity relative to recent trading volume'),
    ('intraday_range', 'Volume and range', '(High_t − Low_t) / Close_t', 'Current trading day', 'No', 'High, Low and Close at t', 'Current intraday price range relative to closing price'),
]
feature_dictionary = pd.DataFrame(dictionary_rows, columns=[
    'feature_name', 'feature_group', 'exact_definition', 'lookback_window', 'annualised',
    'information_available_at_prediction_time', 'expected_interpretation',
])
assert feature_dictionary['feature_name'].tolist() == PREDICTORS, \
    'The feature dictionary must document every predictor in output order.'
feature_dictionary.to_csv(DICTIONARY_PATH, index=False)
print(f'Saved documentation for {len(feature_dictionary)} predictors.')
display(feature_dictionary)

## 4. Date-only merge and unmatched-date investigation

Predictors are merged with the existing feasibility dataset by `Date` only. Raw dates absent from that dataset are investigated as initial feasibility exclusions, the five purged dates or final observations without a complete future target. No target, threshold or sample label is recalculated.

In [ ]:
feature_frame = features.reset_index()
audit_merge = feature_frame[['Date']].merge(
    feasibility[['Date']], on='Date', how='outer', indicator=True, validate='one_to_one'
)
raw_only_dates = pd.DatetimeIndex(audit_merge.loc[audit_merge['_merge'].eq('left_only'), 'Date'])
feasibility_only_dates = pd.DatetimeIndex(audit_merge.loc[audit_merge['_merge'].eq('right_only'), 'Date'])

initial_exclusion_dates = ohlcv.index[ohlcv.index < feasibility['Date'].min()]
final_targetless_dates = ohlcv.index[ohlcv.index > feasibility['Date'].max()]
explained_raw_only = initial_exclusion_dates.union(purged_dates).union(final_targetless_dates)
assert feasibility_only_dates.empty, 'Every feasibility date must have a matching raw feature date.'
assert raw_only_dates.equals(explained_raw_only.sort_values()), \
    'Raw-only dates are not fully explained by initial exclusions, the purge and final targetless observations.'

feasibility_targets = feasibility[['Date', 'future_rv_5d', 'high_volatility', 'sample_period']].copy()
merged = feasibility_targets.merge(feature_frame, on='Date', how='left', validate='one_to_one', indicator=True)
assert merged['_merge'].eq('both').all(), 'Every feasibility observation must match one feature row by Date.'
merged = merged.drop(columns='_merge')

preserved = merged[['Date', 'future_rv_5d', 'high_volatility', 'sample_period']].merge(
    feasibility[['Date', 'future_rv_5d', 'high_volatility', 'sample_period']],
    on='Date', how='inner', validate='one_to_one', suffixes=('_merged', '_original')
)
pd.testing.assert_series_equal(
    preserved['future_rv_5d_merged'], preserved['future_rv_5d_original'], check_names=False, check_exact=True
)
pd.testing.assert_series_equal(
    preserved['high_volatility_merged'], preserved['high_volatility_original'], check_names=False, check_exact=True
)
pd.testing.assert_series_equal(
    preserved['sample_period_merged'], preserved['sample_period_original'], check_names=False, check_exact=True
)

missing_predictor_rows = merged[PREDICTORS].isna().any(axis=1)
removed_dates = pd.DatetimeIndex(merged.loc[missing_predictor_rows, 'Date'])
final_dataset = merged.loc[~missing_predictor_rows, [
    'Date', *PREDICTORS, 'future_rv_5d', 'high_volatility', 'sample_period'
]].copy()

assert final_dataset['Date'].is_monotonic_increasing and final_dataset['Date'].is_unique, \
    'Output dates must be increasing and unique.'
assert not final_dataset['Date'].isin(purged_dates).any(), 'The five purged dates must remain absent.'
assert final_dataset.loc[final_dataset['sample_period'].eq('test'), 'Date'].min() == expected_test_start, \
    'The test start date must remain 10 March 2021.'
assert not final_dataset[PREDICTORS].isna().any().any(), 'No predictor may contain missing values.'
assert np.isfinite(final_dataset[PREDICTORS].to_numpy(dtype=float)).all(), \
    'No predictor may contain infinite values.'
assert (final_dataset[PREDICTORS].nunique() > 1).all(), 'No predictor may be constant.'
assert not final_dataset[['future_rv_5d', 'high_volatility', 'sample_period']].isna().any().any(), \
    'All output rows must contain complete targets and sample labels.'
assert final_dataset['sample_period'].isin(['train', 'test']).all(), \
    'Only the preserved train and test labels are permitted.'

print(f'Feature dates without feasibility rows: {len(raw_only_dates)}')
print('Raw-only dates:', ', '.join(date.date().isoformat() for date in raw_only_dates))
print(f'Feasibility dates without feature rows: {len(feasibility_only_dates)}')
print(f'Initial raw dates excluded by feasibility requirements: {len(initial_exclusion_dates)}')
print(f'Purged dates: {len(purged_dates)}')
print(f'Final raw dates without a complete future target: {len(final_targetless_dates)}')
print(f'Feasibility observations removed for incomplete initial feature lookbacks: {len(removed_dates)}')
print('Removed dates:', ', '.join(date.date().isoformat() for date in removed_dates))
print('The target values and sample labels match the feasibility dataset exactly.')

## 5. Predictor diagnostics and output

Descriptive statistics are reported separately for the training and test predictors. The correlation heatmap uses training predictors only and is diagnostic; no feature is removed solely because it is correlated at this stage. No scaling, normalisation or feature selection is performed.

In [ ]:
train_predictors = final_dataset.loc[final_dataset['sample_period'].eq('train'), PREDICTORS]
test_predictors = final_dataset.loc[final_dataset['sample_period'].eq('test'), PREDICTORS]

print('Training predictor descriptive statistics:')
display(train_predictors.describe().T)
print('Test predictor descriptive statistics:')
display(test_predictors.describe().T)

feature_ranges = pd.DataFrame({
    'training_minimum': train_predictors.min(),
    'training_maximum': train_predictors.max(),
    'test_minimum': test_predictors.min(),
    'test_maximum': test_predictors.max(),
})
print('Predictor ranges by sample period:')
print(feature_ranges.to_string(float_format=lambda value: f'{value:.6f}'))

correlation = train_predictors.corr()
fig, ax = plt.subplots(figsize=(13, 11))
sns.heatmap(
    correlation, cmap='vlag', center=0, vmin=-1, vmax=1, square=True,
    linewidths=0.4, cbar_kws={'label': 'Pearson correlation'}, ax=ax
)
ax.set_title('XLK Training-Predictor Correlation Heatmap')
ax.set_xlabel('Predictor')
ax.set_ylabel('Predictor')
fig.tight_layout()
fig.savefig(HEATMAP_PATH, dpi=300, bbox_inches='tight')
plt.show()

final_dataset.to_csv(FEATURE_PATH, index=False)
print(f'Final training observations: {len(train_predictors):,}')
print(f'Final test observations: {len(test_predictors):,}')
print(f'Feature dataset saved to: {FEATURE_PATH.relative_to(PROJECT_ROOT)}')
print(f'Feature dictionary saved to: {DICTIONARY_PATH.relative_to(PROJECT_ROOT)}')
print(f'Training correlation heatmap saved to: {HEATMAP_PATH.relative_to(PROJECT_ROOT)}')
print('All feature-engineering quality checks passed; no model was trained.')

## Scope and limitations

This initial feature set is intentionally compact. Correlations and distribution shifts are reported for diagnosis rather than used for feature removal. Decisions about scaling, feature selection, validation and modelling belong to later stages and are not undertaken here.